# Single-User Candidate Space

This notebook is the first code-facing walkthrough of the single-user model. It takes one fixed radio scenario and one fixed user demand, runs the single-user search, and then inspects the feasible candidate set that comes back.

The main point is simple. Even when the scenario and user case are fixed, the solver can still admit many different PHY allocations. The notebook therefore does not try to present one universal answer. Instead, it shows how the candidate space is built, what a feasible result table looks like, and how one selected row can be read as a concrete time-frequency-layer allocation.


## 1. Fixed scenario and one explicit user case

We start from the shared single-user study setup. That setup already fixes the common radio assumptions used in this part of the repo, including the carrier setting, bandwidth options, slot window, admitted layer counts, MCS sweep, PRB step, and PA catalog.

One explicit user case is then pinned down by two inputs: distance and required average downlink rate. Once those are fixed, the code can build one scenario object, prepare the corresponding deployment, and define the discrete search space that will be explored in the later cells.


### Terms used in this notebook

A few terms appear often in the tables and plots:

- **PA**: the power amplifier model used for the candidate.
- **BWP**: the bandwidth part that defines the available frequency span for scheduling.
- **PRB**: the number of scheduled physical resource blocks inside that bandwidth part.
- **MCS**: the modulation and coding choice used to map resources to throughput and required SINR.
- **Layers**: the number of spatial layers used by the candidate.
- **Candidate**: one discrete tuple of the form `(PA, BWP, n_PRB, n_slots_on, layers, MCS)`.
- **Feasible candidate**: a candidate that passes the model checks and meets the required rate.

The notebook moves from the prepared scenario, to raw candidate tuples, to the final feasible table for that one user case.


In [1]:
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib import colors
import pandas as pd
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import display

DOC_IMG_DIR = Path.cwd() / "img"
DOC_IMG_DIR.mkdir(parents=True, exist_ok=True)


def export_doc_figure(fig, filename):
    output_path = DOC_IMG_DIR / filename
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    print(f"Saved figure to {output_path}")
    return output_path


sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from single_user_study import (
    build_single_user_scenario,
    preview_single_user_candidates,
    run_single_user_scenario,
    summarize_single_user_scenario,
)

In [2]:
user_case = {
    "distance_m": 300.0,
    "required_rate_bps": 5e6,
}

scenario = build_single_user_scenario(
    distance_m=float(user_case["distance_m"]),
    required_rate_bps=float(user_case["required_rate_bps"]),
)
search_result = run_single_user_scenario(scenario)
search_views = summarize_single_user_scenario(scenario)

candidate_space_view = search_views["candidate_space_view"].copy()
example_candidate_view = search_views["example_candidate_view"].copy()
pa_characteristics = search_views["pa_characteristics"].copy()

candidate_preview = preview_single_user_candidates(scenario, limit=5).copy()
candidate_table = search_result.copy().reset_index(drop=True)
feasible_table = candidate_table.copy()

if feasible_table.empty:
    raise ValueError("The chosen user case produced no feasible candidates.")

best_feasible_row = (
    feasible_table.sort_values(
        [
            "p_dc_avg_total_w",
            "bandwidth_hz",
            "n_prb",
            "n_slots_on",
            "layers",
            "mcs",
            "pa_id",
            "bwp_idx",
        ]
    )
    .reset_index(drop=True)
    .iloc[0]
)

total_slots = int(example_candidate_view.loc[0, "available_slots"])
total_prbs = int(example_candidate_view.loc[0, "available_prbs"])
max_layers = int(example_candidate_view.loc[0, "available_layers"])


## 2. Prepared case, search-space views, and candidate preview

The next outputs make the prepared case concrete.

The first tables summarize the search space that is available for this user case and the PA options that can be used inside it. These are not yet the final feasible results. They describe the bounds of the search.

The preview table shown after them then lists a few raw candidate tuples produced from that setup. This preview is useful because it makes the search variables visible before feasibility filtering is applied. At this stage, the rows are examples from the discrete candidate construction, not yet the final set of admissible solutions.


In [ ]:
display(candidate_space_view)
display(pa_characteristics)


In [ ]:
display(candidate_preview)


## 3. Feasible candidate cloud across PA choices

After the raw tuples have been evaluated, the solver returns the feasible candidate table for this one user case. Each remaining row is one allocation that satisfies the required rate and passes the model checks.

The scatter plot below makes the main point of the notebook visible. A single user request does not collapse into one forced PHY configuration. Even with a fixed scenario, the solver can still admit a cloud of structurally different feasible rows across bandwidth, PRBs, active slots, layers, MCS, and PA choice.

That multiplicity is the reason the later optimization problem exists at all. If the feasible set contained only one row, there would be nothing to compare.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (_pa_id, scenario_label), df_pa in feasible_table.groupby(["pa_id", "scenario_label"], sort=True):
    ax.scatter(
        df_pa["rate_ach_bps"] / 1e6,
        df_pa["p_dc_avg_total_w"],
        s=18,
        alpha=0.7,
        label=scenario_label,
    )

ax.scatter(
    best_feasible_row["rate_ach_bps"] / 1e6,
    best_feasible_row["p_dc_avg_total_w"],
    s=55,
    color="red",
    label="Illustrative feasible row",
)

ax.set_xlabel("Achievable rate (Mbps)")
ax.set_ylabel("Window-averaged total PA DC power (W)")
ax.set_title("Feasible single-user candidate cloud for one fixed user case")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
export_doc_figure(fig, "single_user_resource_allocation_feasible_points.png")
plt.show()

## 4. Reading one feasible allocation

To make the feasible set easier to interpret, one deterministic example row is selected: the feasible candidate with the lowest window-averaged total PA DC power after a stable ordering. This row is not treated as the only meaningful solution. It is simply one reproducible example taken from the larger feasible cloud.

The resource view below reads that row along three axes:

- `n_slots_on` gives the occupied time extent inside the fixed slot window,
- `n_prb` gives the occupied frequency extent inside the chosen BWP,
- `layers` gives the occupied spatial extent.

The 3D block should therefore be read as a compact interpretation aid for one feasible row. It shows how one candidate occupies part of the available time-frequency-layer envelope for the prepared case.


In [ ]:
display(example_candidate_view)

In [ ]:
def cuboid(ax, x, y, z, dx, dy, dz, color, alpha=0.45, edgecolor="black", linewidth=1.2):
    vertices = [
        [x, y, z],
        [x + dx, y, z],
        [x + dx, y + dy, z],
        [x, y + dy, z],
        [x, y, z + dz],
        [x + dx, y, z + dz],
        [x + dx, y + dy, z + dz],
        [x, y + dy, z + dz],
    ]
    faces = [
        [vertices[i] for i in [0, 1, 2, 3]],
        [vertices[i] for i in [4, 5, 6, 7]],
        [vertices[i] for i in [0, 1, 5, 4]],
        [vertices[i] for i in [2, 3, 7, 6]],
        [vertices[i] for i in [1, 2, 6, 5]],
        [vertices[i] for i in [0, 3, 7, 4]],
    ]
    poly = Poly3DCollection(
        faces,
        facecolors=color,
        edgecolor=edgecolor,
        linewidths=linewidth,
        alpha=alpha,
    )
    ax.add_collection3d(poly)


fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")

envelope_color = colors.to_hex(plt.cm.Greys(0.55))
allocation_color = colors.to_hex(plt.cm.Blues(0.68))

cuboid(
    ax,
    0,
    0,
    0,
    total_slots,
    total_prbs,
    max_layers,
    envelope_color,
    alpha=0.08,
    edgecolor="#8a8a8a",
    linewidth=1.0,
)

cuboid(
    ax,
    0,
    0,
    0,
    int(best_feasible_row["n_slots_on"]),
    int(best_feasible_row["n_prb"]),
    int(best_feasible_row["layers"]),
    allocation_color,
    alpha=0.55,
    edgecolor="#1f4e79",
    linewidth=1.4,
)

ax.set_xlabel("Time resources (slots)", labelpad=12)
ax.set_ylabel("Frequency resources (PRBs)", labelpad=14)
ax.zaxis.set_rotate_label(False)
ax.set_zlabel("Spatial layers", rotation=90, labelpad=18)

ax.set_xlim(0, total_slots)
ax.set_ylim(total_prbs, 0)
ax.set_zlim(0, max_layers)
ax.set_xticks(range(0, total_slots + 1, max(1, total_slots // 10)))
ax.set_yticks(range(0, total_prbs + 1, max(1, total_prbs // 6)))
ax.set_zticks(range(0, max_layers + 1, 1))
ax.set_title(
    "Illustrative feasible PHY allocation\n"
    f"{best_feasible_row['scenario_label']} | "
    f"MCS={int(best_feasible_row['mcs'])} | "
    f"rate={best_feasible_row['rate_ach_bps'] / 1e6:.1f} Mbps | "
    f"power={best_feasible_row['p_dc_avg_total_w']:.2f} W"
)
ax.view_init(elev=22, azim=-62)
plt.tight_layout()
export_doc_figure(fig, "single_user_resource_allocation_candidate.png")
plt.show()

This notebook establishes the starting point for the single-user model at code level: one fixed scenario, one explicit user case, one constructed candidate space, and one resulting feasible cloud.

The next notebook, `single_user_downlink_power_optimization.ipynb`, builds on that result and studies how the preferred feasible solutions change as the user demand changes.
